![Your 6-week roadmap](../assets/roadmap/week1.png)

# Week 1 Exercise: Multiple-Choice Questions in the Exam Schema

Great news from Northbridge Institute: Asha liked your extractor so much that she's redesigning next term's midterm around it. 🎉 There's one catch. She wants to add a new **Section D of multiple-choice questions** (MCQs), and our exam schema can't really hold them.

Look again at `gradewise/schemas/exam.py`. `QuestionType` already allows `"mcq"`, but a `Question` has nowhere to put the **options** (A, B, C, D) or the **correct answer**. Today you fix that.

**The rule:** don't edit `gradewise/schemas/`. The rest of GradeWise (the extractor, the database, the week 3 graph) depends on those classes. Instead you'll **extend** them in this notebook, by subclassing, the way you'd add a feature to a library you don't own. If your design is right, everything that works with an `ExamPaper` keeps working with yours.

### 🎯 Your mission
1. **`MCQOption`**: a `label` (`"A"`) and its `text`
2. **`MCQQuestion(Question)`**: adds `options` and `correct_option`, with these rules:
   - an `mcq` question has at least 2 options, unique labels, and a `correct_option` that is one of them
   - any other question type has **no** options and **no** `correct_option`
3. **`MCQExamPaper(ExamPaper)`**: its questions are `MCQQuestion`s, and its `problems()` adds one more check:
   an MCQ must have exactly **one** criterion, worth all its marks (MCQs are all-or-nothing)
4. **`mark_mcq(question, chosen)`**: marks an MCQ answer with plain Python, no LLM needed
5. Extract Asha's draft Section D with `with_structured_output(MCQExamPaper)`, rehearsing for free first

Ten automatic checks at the end of Part 2 tell you how you're doing. They never crash, so run them as often as you like.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Work through the <code>✋ TODO</code> cells in order. Stuck? The Day 3 notebook has every technique you need: subclassing a Pydantic model, <code>Field</code>, <code>model_validator</code> and a <code>problems()</code> list. The reference solution is in <code>week1/solutions/week1_EXERCISE_SOLUTION.ipynb</code>, but give it a real go first!</span>
        </td>
    </tr>
</table>

In [ ]:
# Imports and environment first, all in one place

import json
import os

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.runnables import RunnableLambda
from pydantic import BaseModel, ValidationError

from gradewise.config import DATA_DIR, settings
from gradewise.exam.extract import SYSTEM_PROMPT
from gradewise.schemas import ExamPaper, Question

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"✅ OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("❌ OpenAI API Key not set. Please see Part 5 of the setup guide (and remember to SAVE .env!)")
    print("   Using another provider? That's fine: see setup/SWAP_PROVIDER.md")

print(f"🧠 Everyday model from your .env: {settings.chat_model}")

## Part 1: The material

Here's Asha's draft of Section D: the questions, her marking scheme and the methodology for MCQs.

In [ ]:
# 💡 Asha's draft of next term's new Section D, exactly as it will be printed
SECTION_D_PAPER = """## **Section D: Multiple choice**
Circle ONE answer for each question.

**Question 11**
Which of the following is NOT one of the four necessary conditions for a deadlock? **[1 mark]**
A. Mutual exclusion    B. Hold and wait    C. Starvation    D. Circular wait

**Question 12**
In the TCP/IP model, which layer does TCP belong to? **[1 mark]**
A. Network    B. Transport    C. Data link    D. Application

**Question 13**
Which normal form removes partial dependencies on a composite key? **[1 mark]**
A. 1NF    B. 2NF    C. 3NF    D. BCNF
"""

SECTION_D_SCHEME = """## **Section D: multiple choice [3 marks]**
Question 11: C (1 mark). Question 12: B (1 mark). Question 13: B (1 mark).
Multiple-choice questions are all-or-nothing: no partial credit. If more than one option is circled, award 0.
If an answer is crossed out and another circled, mark the circled one."""

SECTION_D_METHOD = """**Multiple-choice questions.** Award the mark only if exactly one option is chosen and it is
the correct one. Ignore working or notes written beside the options. Never award part of a mark."""

And some test data, shaped the way a model might return it. A few are deliberately broken.

In [ ]:
# 💡 Test data, as a model might return it. Use these to check your classes.
GOOD_MCQ = {
    "id": "Q11",
    "number": "11",
    "question_type": "mcq",
    "max_marks": 1,
    "text": "Which of the following is NOT one of the four necessary conditions for a deadlock?",
    "options": [
        {"label": "A", "text": "Mutual exclusion"},
        {"label": "B", "text": "Hold and wait"},
        {"label": "C", "text": "Starvation"},
        {"label": "D", "text": "Circular wait"},
    ],
    "correct_option": "C",
    "criteria": [{"id": "Q11.c1", "description": "Chooses C (starvation) and only C.", "marks": 1}],
    "methodology": "All-or-nothing. Award the mark only if exactly one option is chosen and it is correct.",
}

WRONG_ANSWER_KEY = {**GOOD_MCQ, "correct_option": "E"}  # ❌ there is no option E
ONE_OPTION = {**GOOD_MCQ, "options": GOOD_MCQ["options"][:1], "correct_option": "A"}  # ❌ not a real choice
DUPLICATE_LABELS = {**GOOD_MCQ, "options": GOOD_MCQ["options"][:2] + [{"label": "A", "text": "Starvation"}]}  # ❌
WRITTEN_WITH_OPTIONS = {**GOOD_MCQ, "question_type": "short"}  # ❌ a written question can't have options
WRITTEN = {k: v for k, v in GOOD_MCQ.items() if k not in ("options", "correct_option")} | {"question_type": "short"}
TWO_CRITERIA = {  # ✅ valid shape, but problems() should complain: MCQs are all-or-nothing
    **GOOD_MCQ,
    "criteria": [
        {"id": "Q11.c1", "description": "Chooses C.", "marks": 0.5},
        {"id": "Q11.c2", "description": "Says why.", "marks": 0.5},
    ],
}


def mini_exam(*questions: dict) -> dict:
    """Wrap questions in a tiny exam whose total matches."""
    return {
        "exam_id": "mcq_demo",
        "title": "Section D",
        "subject": "CS201",
        "questions": list(questions),
        "total_marks": sum(q["max_marks"] for q in questions),
    }


EXPECTED_EXAM = json.loads((DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json").read_text(encoding="utf-8"))

## Part 2: Extend the schema

### 2.1 `MCQOption` and `MCQQuestion`

`MCQQuestion` **inherits** every field of `Question` (`id`, `text`, `criteria`, ...), so you only add what's new. Give `options` and `correct_option` **defaults** (an empty list and `None`): that's what lets a written question, like every question in the CS201 midterm, still validate.

In [ ]:
class MCQOption(BaseModel):
    """One answer option of a multiple-choice question."""

    # ✋ TODO: add a `label` field (e.g. "A") and a `text` field, each with a Field(description=...)
    #          Remember: the descriptions are instructions the MODEL reads.
    pass


class MCQQuestion(Question):
    """A Question that can also be multiple choice (question_type "mcq")."""

    # ✋ TODO: add `options: list[MCQOption]` with an empty-list default (Field(default_factory=list, ...))
    # ✋ TODO: add `correct_option: str | None`, defaulting to None

    # ✋ TODO: add a @model_validator(mode="after") that raises ValueError when:
    #          - a non-mcq question has options or a correct_option
    #          - an mcq has fewer than 2 options, or duplicate labels
    #          - an mcq's correct_option isn't one of its labels
    #          💡 Compare labels with .strip().upper(), so " c" and "C" match.
    pass

### 2.2 `MCQExamPaper`

Override the `questions` field so it holds `MCQQuestion`s, and extend `problems()`: call `super().problems()` to keep every existing check, then add yours.

In [ ]:
class MCQExamPaper(ExamPaper):
    """An ExamPaper whose questions may be multiple choice."""

    # ✋ TODO: override `questions` so it is a list[MCQQuestion] (keep min_length=1)
    # ✋ TODO: override problems(): start from super().problems(), then add an issue mentioning the question ID
    #          for every mcq that doesn't have exactly ONE criterion worth all of its max_marks
    pass

### 2.3 Marking an MCQ without an LLM

Here's a lovely consequence of good structure: once the correct option is *data*, marking an MCQ is one line of Python. No model, no cost, no randomness. The marking scheme's rule "more than one option circled earns 0" belongs here too.

In [ ]:
def mark_mcq(question: MCQQuestion, chosen: str) -> float:
    """Full marks for exactly one correct option, 0 otherwise. Case and spaces don't matter."""
    # ✋ TODO: raise ValueError if the question isn't an mcq
    # ✋ TODO: split `chosen` on commas/spaces into a set of upper-case labels ("A, C" → {"A", "C"})
    # ✋ TODO: return question.max_marks if that set is exactly {correct_option}, else 0.0
    raise NotImplementedError("✋ TODO: write mark_mcq")

### 2.4 The checks

Run this cell whenever you like. Aim for ten green ticks.

In [ ]:
def check(name: str, test) -> None:
    """Run one test and print ✅ or ❌. Never raises, so you can run this cell at any time."""
    try:
        passed = bool(test())
        print(("✅ " if passed else "❌ ") + name)
    except Exception as exc:  # noqa: BLE001  💡 a crash counts as a fail, with the reason
        print(f"❌ {name}  ({type(exc).__name__}: {str(exc).splitlines()[0][:80]})")


def rejects(model, data) -> bool:
    try:
        model.model_validate(data)
    except ValidationError:
        return True
    return False


check("1. a good MCQ validates and keeps its 4 options", lambda: len(MCQQuestion.model_validate(GOOD_MCQ).options) == 4)
check("2. the correct option must be one of the labels", lambda: rejects(MCQQuestion, WRONG_ANSWER_KEY))
check("3. an MCQ needs at least 2 options", lambda: rejects(MCQQuestion, ONE_OPTION))
check("4. option labels must be unique", lambda: rejects(MCQQuestion, DUPLICATE_LABELS))
check("5. a written question can't have options", lambda: rejects(MCQQuestion, WRITTEN_WITH_OPTIONS))
check(
    "6. a written question without options still validates", lambda: MCQQuestion.model_validate(WRITTEN).options == []
)
check(
    "7. the CS201 midterm still loads, with no problems",
    lambda: MCQExamPaper.model_validate(EXPECTED_EXAM).problems() == [],
)
check(
    "8. problems() flags an MCQ that isn't one all-or-nothing criterion",
    lambda: any("Q11" in p for p in MCQExamPaper.model_validate(mini_exam(TWO_CRITERIA)).problems()),
)
check(
    "9. mark_mcq: ' c' earns full marks, 'B' earns 0, 'A, C' earns 0",
    lambda: [mark_mcq(MCQQuestion.model_validate(GOOD_MCQ), a) for a in (" c", "B", "A, C")] == [1, 0, 0],
)
check(
    "10. the JSON schema the model sees includes options and correct_option",
    lambda: {"options", "correct_option"} <= set(json.dumps(MCQExamPaper.model_json_schema()).split('"')),
)

## Part 3: Extract Section D

Now let the model do the work. We reuse the package's `SYSTEM_PROMPT` (all ten rules still apply) and add an eleventh rule for MCQs:

In [ ]:
MCQ_RULES = """
11. Multiple-choice questions have question_type "mcq". Copy every option into `options`, with its letter as
    `label` and its wording as `text`. Put the letter of the right answer, from the MARKING SCHEME, in
    `correct_option`. Give each MCQ exactly ONE criterion worth all of its marks. Written questions have no
    options and a null correct_option.
"""

mcq_messages = [
    SystemMessage(SYSTEM_PROMPT + MCQ_RULES),  # 💡 the package's checklist, plus one new rule
    HumanMessage(
        'Exam ID: "mcq_demo"\n\n'
        f"=== QUESTION PAPER ===\n{SECTION_D_PAPER}\n\n"
        f"=== MARKING SCHEME ===\n{SECTION_D_SCHEME}\n\n"
        f"=== MARKING METHODOLOGY ===\n{SECTION_D_METHOD}"
    ),
]
print(mcq_messages[1].content[:400], "…")

First a **dress rehearsal** with a pretend model, like on Day 3: it hands back a canned answer, so you can test the plumbing for free.

In [ ]:
class PretendModel:
    """with_structured_output() returns a canned answer, validated against the schema you pass."""

    def __init__(self, answer: dict):
        self.answer = answer

    def with_structured_output(self, schema, **kwargs):
        return RunnableLambda(lambda messages: schema.model_validate(self.answer))


canned = mini_exam(GOOD_MCQ)

# ✋ TODO (after Part 2 passes): uncomment these lines and run them
# rehearsal = PretendModel(canned).with_structured_output(MCQExamPaper).invoke(mcq_messages)
# print_exam_summary(rehearsal)
# print("Q11 answer key:", rehearsal.questions[0].correct_option)

And the real thing. Section D is short, so this costs well under a cent.

In [ ]:
# ✋ TODO: create the model with chat_model(temperature=0), call with_structured_output(MCQExamPaper),
#          invoke it on mcq_messages, set exam_id to "mcq_demo" with model_copy, and print_exam_summary() it.
#          Then print each question's options and correct_option.

## Recap, and stretch goals

When all ten checks are green and Section D extracts cleanly, you've extended GradeWise's schema the professional way: **without editing the package**, and without breaking a single existing exam. 🎉

**Stretch goals**
- Mark Sam's Section D with `mark_mcq`: he circled `{"Q11": "C", "Q12": "b", "Q13": "B, C"}`. What does he score?
- Combine the midterm and Section D into one `MCQExamPaper` with 53 marks. Does `problems()` stay clean?
- Use the package's self-correcting loop with your schema: `extract_with_retry(SECTION_D_PAPER, SECTION_D_SCHEME, SECTION_D_METHOD, exam_id="mcq_demo", schema=MCQExamPaper)` (from `gradewise.exam`). Your `problems()` now drives the retries!
- Read about Pydantic **discriminated unions**. Would two separate classes (written vs MCQ) be better than optional fields? What would the model's JSON schema look like?

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;">Want to share your solution? Put a copy in <code>community_contributions/</code> (outputs cleared, please) and open a pull request. Compare with <code>week1/solutions/week1_EXERCISE_SOLUTION.ipynb</code> when you're done.</span>
        </td>
    </tr>
</table>